# WAF 1단계 — 룰 기반(Coraza + OWASP CRS) vs AI 탐지 비교

**목표**: 같은 HTTP 요청 데이터로 룰 기반 WAF와 AI 모델의 탐지율·오탐률을 나란히 비교한다.

| 구분 | 내용 |
|---|---|
| 데이터 | HTTP DATASET CSIC 2010 (정상 72,000 / 비정상 25,065 요청) |
| 룰 기반 | OWASP Coraza v3.8.1 + OWASP CRS 4.25 LTS, 편집증 수준(Paranoia Level) 1–4 |
| AI 모델 A | 문자 n-gram TF-IDF + 로지스틱 회귀 (CPU, 설명 가능한 기준선) |
| AI 모델 B | 문자 단위 CNN (GPU) |

**실행 전**: 런타임 → 런타임 유형 변경 → **T4 GPU** 선택 후, 위에서부터 차례로 실행합니다.
전체 실행 시간은 약 15–25분입니다 (Go·Coraza 빌드 3–5분, CRS 채점 5–10분, 학습 수 분).

## 1. 환경 확인

In [ ]:
import os, sys, re, json, time, random, shutil, subprocess, platform, hashlib
IN_COLAB = 'google.colab' in sys.modules
WORK = '/content/waf-lab' if IN_COLAB else os.path.abspath(os.environ.get('WAF_LAB_DIR', 'waf-lab'))
os.makedirs(WORK, exist_ok=True)
os.chdir(WORK)
RESULTS = os.path.join(WORK, 'results')
os.makedirs(RESULTS, exist_ok=True)

def sh(cmd, check=True, show=True):
    """셸 명령 실행 (Colab·로컬 공통)"""
    if show:
        print('$', cmd)
    r = subprocess.run(cmd, shell=True, text=True, capture_output=True)
    out = (r.stdout + r.stderr).strip()
    if show and out:
        print(out[-4000:])
    if check and r.returncode != 0:
        raise RuntimeError(f'명령 실패 (코드 {r.returncode}): {cmd}')
    return r

import numpy as np, pandas as pd, sklearn, torch
SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
VERSIONS = {
    'python': platform.python_version(), 'numpy': np.__version__, 'pandas': pd.__version__,
    'scikit-learn': sklearn.__version__, 'torch': torch.__version__,
    'gpu': torch.cuda.get_device_name(0) if DEVICE == 'cuda' else 'none (CPU)',
}
print(json.dumps(VERSIONS, indent=2, ensure_ascii=False))
if DEVICE != 'cuda':
    print('※ GPU가 없습니다. 런타임 유형을 T4 GPU로 바꾸면 모델 B 학습이 훨씬 빠릅니다 (CPU로도 실행은 됨).')

## 2. 데이터 받기 (CSIC 2010)
공식 페이지(tic.itefi.csic.es/dataset)의 파일과 같은 내용을 담은 공개 저장소에서 받습니다.
요청 수(36,000 / 36,000 / 25,065)로 원본과 같은지 확인합니다.

In [ ]:
DATA = os.path.join(WORK, 'csic')
FILES = ['normalTrafficTraining.txt', 'normalTrafficTest.txt', 'anomalousTrafficTest.txt']
MIRRORS = [
    'https://github.com/baksakal/HTTP-DATASET-CSIC-2010-MACHINE-LEARNING-GUI-AND-SERVER',
    'https://gitlab.fing.edu.uy/gsi/web-application-attacks-datasets',   # csic_2010/ 하위 폴더
]
if not all(os.path.exists(os.path.join(DATA, f)) for f in FILES):
    os.makedirs(DATA, exist_ok=True)
    for url in MIRRORS:
        tmp = os.path.join(WORK, '_mirror')
        shutil.rmtree(tmp, ignore_errors=True)
        if sh(f'git clone -q --depth 1 {url} {tmp}', check=False).returncode != 0:
            continue
        for f in FILES:
            for cand in (os.path.join(tmp, f), os.path.join(tmp, 'csic_2010', f)):
                if os.path.exists(cand):
                    shutil.copy(cand, os.path.join(DATA, f))
        shutil.rmtree(tmp, ignore_errors=True)
        if all(os.path.exists(os.path.join(DATA, f)) for f in FILES):
            print('받은 곳:', url)
            break
missing = [f for f in FILES if not os.path.exists(os.path.join(DATA, f))]
if missing:
    raise SystemExit(f'데이터를 받지 못했습니다: {missing}\n'
                     f'공식 페이지에서 직접 받아 {DATA} 폴더에 올린 뒤 다시 실행하세요.')
for f in FILES:
    p = os.path.join(DATA, f)
    print(f'{f:28} {os.path.getsize(p):>11,} bytes  sha256={hashlib.sha256(open(p,"rb").read()).hexdigest()[:16]}…')

## 3. 요청 파싱과 중복 제거
CSIC 원본에는 **같은 요청이 여러 번** 들어 있습니다 (예: `GET /tienda1/index.jsp`가 2,000번).
중복을 그대로 두면 학습용과 시험용에 같은 요청이 섞여 점수가 부풀려지므로,
`메서드 + URL + 본문`이 같은 요청은 하나만 남깁니다.

In [ ]:
REQ_START = re.compile(r'^(GET|POST|PUT|DELETE|HEAD|OPTIONS|PATCH) (\S+) (HTTP/\d\.\d)$')

def parse_csic(path, label, source):
    lines = open(path, encoding='latin-1').read().replace('\r\n', '\n').split('\n')
    out, i, n = [], 0, len(lines)
    while i < n:
        m = REQ_START.match(lines[i])
        if not m:
            i += 1
            continue
        method, target, _ = m.groups()
        headers = {}
        i += 1
        while i < n and lines[i] != '':
            if ':' in lines[i]:
                k, v = lines[i].split(':', 1)
                headers[k.strip().lower()] = v.strip()
            i += 1
        body = ''
        if int(headers.get('content-length', '0') or 0) > 0:
            while i < n and lines[i] == '':
                i += 1
            if i < n and not REQ_START.match(lines[i]):
                body = lines[i]
                i += 1
        out.append(dict(label=label, source=source, method=method,
                        url=re.sub(r'^https?://[^/]+', '', target), body=body,
                        content_type=headers.get('content-type', '') if body else ''))
    return out

rows = (parse_csic(os.path.join(DATA, FILES[0]), 0, 'normal_train')
        + parse_csic(os.path.join(DATA, FILES[1]), 0, 'normal_test')
        + parse_csic(os.path.join(DATA, FILES[2]), 1, 'anomalous'))
raw = pd.DataFrame(rows)
raw['key'] = raw.method + ' ' + raw.url + ' ' + raw.body
print('원본 요청 수:'); print(raw.groupby('source').size().to_string())
assert raw.groupby('source').size().to_dict() == {'anomalous': 25065, 'normal_test': 36000, 'normal_train': 36000}, '원본 요청 수가 다릅니다'

conflict = set(raw.key[raw.label == 0]) & set(raw.key[raw.label == 1])
df = raw.drop_duplicates('key').reset_index(drop=True)
df = df[~df.key.isin(conflict)].reset_index(drop=True)
df['id'] = np.arange(len(df))
DATA_STATS = {
    'raw_total': int(len(raw)), 'raw_normal': int((raw.label == 0).sum()), 'raw_anomalous': int((raw.label == 1).sum()),
    'unique_total': int(len(df)), 'unique_normal': int((df.label == 0).sum()), 'unique_anomalous': int((df.label == 1).sum()),
    'label_conflicts_removed': len(conflict),
}
print(json.dumps(DATA_STATS, indent=2))
print(f'→ 중복 제거로 {len(raw) - len(df):,}건({(len(raw) - len(df)) / len(raw):.0%}) 제외')

## 4. 학습 / 검증 / 시험 분할
중복을 뺀 요청을 정상·비정상 비율을 유지한 채 **60 / 20 / 20**으로 나눕니다.
- 학습(train): AI 모델 학습
- 검증(val): AI 판정 기준값(threshold) 선택 — "오탐률 1%" 같은 운영 기준을 여기서 정함
- 시험(test): **모든 비교는 이 집합에서만** (룰도 AI도 처음 보는 요청)

In [ ]:
from sklearn.model_selection import train_test_split
idx_train, idx_tmp = train_test_split(df.index, test_size=0.4, stratify=df.label, random_state=SEED)
idx_val, idx_test = train_test_split(idx_tmp, test_size=0.5, stratify=df.label[idx_tmp], random_state=SEED)
df['split'] = 'train'
df.loc[idx_val, 'split'] = 'val'
df.loc[idx_test, 'split'] = 'test'
print(df.groupby(['split', 'label']).size().unstack().rename(columns={0: 'normal', 1: 'anomalous'}))
df[['id', 'method', 'url', 'body', 'content_type']].to_csv('requests.csv', index=False)

## 5. Coraza + OWASP CRS 준비
Coraza는 Go 라이브러리라, 요청 CSV를 한꺼번에 검사하는 작은 Go 프로그램(`crsscan`)을 만들어 씁니다.
CRS 규칙은 `coraza-coreruleset` 모듈(v4.25.0)에 들어 있는 것을 그대로 사용합니다.

In [ ]:
GO_VERSION = '1.25.14'
CORAZA_VERSION = 'v3.8.1'
CRS_MODULE_VERSION = 'v4.25.0'
SCAN = os.path.join(WORK, 'crsscan')
os.makedirs(SCAN, exist_ok=True)

def find_go():
    for g in ('/usr/local/go/bin/go', shutil.which('go') or ''):
        if g and os.path.exists(g):
            v = subprocess.run([g, 'version'], capture_output=True, text=True).stdout
            m = re.search(r'go(\d+)\.(\d+)', v)
            if m and (int(m.group(1)), int(m.group(2))) >= (1, 25):
                return g
    return None

GO = os.environ.get('GO_BIN') or find_go()
if not GO:
    sh(f'curl -fsSL -o /tmp/go.tgz https://go.dev/dl/go{GO_VERSION}.linux-amd64.tar.gz')
    sh('rm -rf /usr/local/go && tar -C /usr/local -xzf /tmp/go.tgz')
    GO = '/usr/local/go/bin/go'
sh(f'{GO} version')

In [ ]:
MAIN_GO = r'''
// crsscan: CSV(요청) → Coraza + OWASP CRS 로 채점 → CSV(결과)
// 입력 열: id,method,url,body,content_type
package main

import (
	"encoding/csv"
	"flag"
	"fmt"
	"io"
	"log"
	"os"
	"sort"
	"strconv"
	"strings"
	"sync"

	coreruleset "github.com/corazawaf/coraza-coreruleset/v4"
	"github.com/corazawaf/coraza/v3"
	"github.com/corazawaf/coraza/v3/types"
)

type result struct {
	id      string
	blocked bool
	status  int
	score   int
	ruleIDs []int
}

// CRS 탐지 규칙만 남김 (초기화·편집증 수준 건너뛰기·집계·보고 규칙 제외)
func isDetectionRule(id int) bool {
	if id < 911000 || id >= 980000 || id%1000 < 100 {
		return false
	}
	p := id / 1000
	return p != 949 && p != 959
}

// CRS 기본 점수표: CRITICAL 5, ERROR 4, WARNING 3, NOTICE 2
func severityScore(s types.RuleSeverity) int {
	switch s {
	case types.RuleSeverityCritical:
		return 5
	case types.RuleSeverityError:
		return 4
	case types.RuleSeverityWarning:
		return 3
	case types.RuleSeverityNotice:
		return 2
	}
	return 0
}

func main() {
	in := flag.String("in", "requests.csv", "input csv")
	out := flag.String("out", "crs_out.csv", "output csv")
	pl := flag.Int("pl", 1, "paranoia level 1-4")
	workers := flag.Int("workers", 4, "parallel workers")
	flag.Parse()

	directives := fmt.Sprintf(`Include @coraza.conf-recommended
SecRuleEngine On
SecRequestBodyAccess On
SecResponseBodyAccess Off
SecAuditEngine Off
Include @crs-setup.conf.example
SecAction "id:900000,phase:1,pass,t:none,nolog,setvar:tx.blocking_paranoia_level=%d"
SecAction "id:900001,phase:1,pass,t:none,nolog,setvar:tx.detection_paranoia_level=%d"
Include @owasp_crs/*.conf`, *pl, *pl)
	waf, err := coraza.NewWAF(coraza.NewWAFConfig().WithRootFS(coreruleset.FS).WithDirectives(directives))
	if err != nil {
		log.Fatalf("WAF init: %v", err)
	}

	f, err := os.Open(*in)
	if err != nil {
		log.Fatal(err)
	}
	r := csv.NewReader(f)
	r.FieldsPerRecord = -1
	header, err := r.Read()
	if err != nil {
		log.Fatal(err)
	}
	col := map[string]int{}
	for i, h := range header {
		col[h] = i
	}
	jobs := make(chan []string, 256)
	results := make(chan result, 256)
	var wg sync.WaitGroup
	for w := 0; w < *workers; w++ {
		wg.Add(1)
		go func() {
			defer wg.Done()
			for rec := range jobs {
				results <- scan(waf, rec[col["id"]], rec[col["method"]], rec[col["url"]], rec[col["body"]], rec[col["content_type"]])
			}
		}()
	}
	go func() {
		for {
			rec, err := r.Read()
			if err == io.EOF {
				break
			}
			if err != nil {
				log.Fatal(err)
			}
			jobs <- rec
		}
		close(jobs)
		wg.Wait()
		close(results)
	}()

	of, err := os.Create(*out)
	if err != nil {
		log.Fatal(err)
	}
	defer of.Close()
	w := csv.NewWriter(of)
	w.Write([]string{"id", "blocked", "status", "score", "rule_ids"})
	n := 0
	for res := range results {
		ids := make([]string, len(res.ruleIDs))
		for i, v := range res.ruleIDs {
			ids[i] = strconv.Itoa(v)
		}
		w.Write([]string{res.id, strconv.FormatBool(res.blocked), strconv.Itoa(res.status), strconv.Itoa(res.score), strings.Join(ids, " ")})
		n++
	}
	w.Flush()
	log.Printf("done: %d requests -> %s (PL%d)", n, *out, *pl)
}

func scan(waf coraza.WAF, id, method, url, body, ctype string) result {
	tx := waf.NewTransaction()
	defer tx.Close()
	res := result{id: id}
	tx.ProcessConnection("10.0.0.1", 40000, "10.0.0.2", 80)
	tx.ProcessURI(url, method, "HTTP/1.1")
	tx.AddRequestHeader("Host", "localhost:8080")
	tx.AddRequestHeader("User-Agent", "Mozilla/5.0 (compatible; Konqueror/3.5; Linux) KHTML/3.5.8 (like Gecko)")
	tx.AddRequestHeader("Accept", "text/xml,application/xml,application/xhtml+xml,text/html;q=0.9,text/plain;q=0.8,image/png,*/*;q=0.5")
	if body != "" {
		if ctype == "" {
			ctype = "application/x-www-form-urlencoded"
		}
		tx.AddRequestHeader("Content-Type", ctype)
		tx.AddRequestHeader("Content-Length", strconv.Itoa(len(body)))
	}
	var it *types.Interruption
	if it = tx.ProcessRequestHeaders(); it == nil && body != "" {
		if it2, _, err := tx.WriteRequestBody([]byte(body)); err == nil && it2 != nil {
			it = it2
		}
	}
	if it == nil {
		it, _ = tx.ProcessRequestBody()
	}
	if it != nil {
		res.blocked = true
		res.status = it.Status
	}
	for _, mr := range tx.MatchedRules() {
		rid := mr.Rule().ID()
		if !isDetectionRule(rid) {
			continue
		}
		res.ruleIDs = append(res.ruleIDs, rid)
		res.score += severityScore(mr.Rule().Severity())
	}
	sort.Ints(res.ruleIDs)
	return res
}
'''
BIN = os.path.join(SCAN, 'crsscan')
if not os.path.exists(BIN):
    open(os.path.join(SCAN, 'main.go'), 'w').write(MAIN_GO)
    if not os.path.exists(os.path.join(SCAN, 'go.mod')):
        sh(f'cd {SCAN} && {GO} mod init crsscan')
    sh(f'cd {SCAN} && {GO} get github.com/corazawaf/coraza/v3@{CORAZA_VERSION} '
       f'github.com/corazawaf/coraza-coreruleset/v4@{CRS_MODULE_VERSION} && {GO} mod tidy', show=False)
    sh(f'cd {SCAN} && {GO} build -o crsscan .')
print('crsscan:', BIN, f'{os.path.getsize(BIN)/1e6:.1f} MB')

# 동작 확인: 정상 1건, SQL 인젝션 1건
pd.DataFrame([dict(id=1, method='GET', url='/tienda1/index.jsp', body='', content_type=''),
              dict(id=2, method='GET', url="/tienda1/publico/anadir.jsp?id=1'+OR+'1'='1", body='', content_type='')]
             ).to_csv('/tmp/smoke.csv', index=False)
sh(f'{BIN} -in /tmp/smoke.csv -out /tmp/smoke_out.csv -pl 1')
print(open('/tmp/smoke_out.csv').read())

## 6. CRS 채점 (편집증 수준 PL1–PL4)
편집증 수준이 높을수록 더 많은 규칙을 켜서 **더 많이 잡지만 오탐도 늘어납니다**.
기본값은 PL1이고, 차단 기준은 CRS 기본값(이상 점수 5점 이상)입니다.
검증·시험 집합만 채점합니다 (룰은 학습이 필요 없음).

In [ ]:
eval_df = df[df.split.isin(['val', 'test'])]
eval_df[['id', 'method', 'url', 'body', 'content_type']].to_csv('eval_requests.csv', index=False)
workers = os.cpu_count() or 2
CRS_SPEED = {}
for pl in (1, 2, 3, 4):
    t0 = time.time()
    sh(f'{BIN} -in eval_requests.csv -out crs_pl{pl}.csv -pl {pl} -workers {workers}')
    sec = time.time() - t0
    CRS_SPEED[f'PL{pl}'] = round(len(eval_df) / sec, 1)
    c = pd.read_csv(f'crs_pl{pl}.csv', dtype={'rule_ids': str}).fillna({'rule_ids': ''})
    c = c.rename(columns={'blocked': f'crs{pl}_block', 'score': f'crs{pl}_score', 'rule_ids': f'crs{pl}_rules'})
    df = df.drop(columns=[x for x in df.columns if x.startswith(f'crs{pl}_')]).merge(
        c[['id', f'crs{pl}_block', f'crs{pl}_score', f'crs{pl}_rules']], on='id', how='left')
    print(f'PL{pl}: {len(eval_df):,}건 {sec:.1f}초 ({CRS_SPEED[f"PL{pl}"]:,} 요청/초, CPU {workers}개)')

## 7. AI 모델 A — 문자 n-gram TF-IDF + 로지스틱 회귀
요청 문자열(`메서드 URL 본문`)을 2–5글자 조각으로 잘라 빈도를 세고, 선형 모델로 판정합니다.
빠르고, 어떤 글자 조각이 판정에 영향을 줬는지 볼 수 있는 **기준선** 모델입니다.

In [ ]:
from urllib.parse import unquote_plus
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression

def to_text(r):
    # URL 인코딩을 한 번 풀어 공격 문자열이 드러나게 함 (원문도 함께 둠)
    s = f'{r.method} {r.url} {r.body}'
    return s + ' || ' + unquote_plus(s, encoding='latin-1')

df['text'] = df.apply(to_text, axis=1)
tr, va, te = (df.split == 'train'), (df.split == 'val'), (df.split == 'test')

t0 = time.time()
vec = TfidfVectorizer(analyzer='char', ngram_range=(2, 5), min_df=2, max_features=300_000, sublinear_tf=True, lowercase=True)
Xtr = vec.fit_transform(df.text[tr])
lr = LogisticRegression(C=4.0, max_iter=3000, class_weight='balanced')
lr.fit(Xtr, df.label[tr])
TRAIN_TIME = {'A_tfidf_lr': round(time.time() - t0, 1)}
t0 = time.time()
df['a_score'] = lr.predict_proba(vec.transform(df.text))[:, 1]
A_SPEED = round(len(df) / (time.time() - t0), 1)
print(f'학습 {TRAIN_TIME["A_tfidf_lr"]}초, 판정 속도 {A_SPEED:,} 요청/초, 특성 {Xtr.shape[1]:,}개')

# 비정상 쪽으로 가장 크게 작용하는 글자 조각
feat = np.array(vec.get_feature_names_out())
top = np.argsort(lr.coef_[0])[::-1][:25]
print('비정상 판정에 영향이 큰 조각:', [feat[i] for i in top])

## 8. AI 모델 B — 문자 단위 CNN (GPU)
요청의 각 바이트를 숫자로 바꿔 1차원 합성곱 신경망으로 판정합니다.
규칙이나 특성을 사람이 정하지 않고, 공격 문자열의 모양을 직접 학습합니다.

In [ ]:
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset

MAX_LEN = 512
EPOCHS = int(os.environ.get('WAF_LAB_EPOCHS', 6))

def encode(texts):
    arr = np.zeros((len(texts), MAX_LEN), dtype=np.int64)
    for i, t in enumerate(texts):
        b = t.encode('latin-1', errors='replace')[:MAX_LEN]
        arr[i, :len(b)] = np.frombuffer(b, dtype=np.uint8).astype(np.int64) + 1  # 0 = 빈칸
    return torch.from_numpy(arr)

class CharCNN(nn.Module):
    def __init__(self):
        super().__init__()
        self.emb = nn.Embedding(257, 48, padding_idx=0)
        self.convs = nn.ModuleList([nn.Conv1d(48, 128, k, padding=k // 2) for k in (3, 5, 7)])
        self.head = nn.Sequential(nn.Dropout(0.3), nn.Linear(384, 128), nn.ReLU(), nn.Linear(128, 1))
    def forward(self, x):
        e = self.emb(x).transpose(1, 2)
        h = torch.cat([torch.relu(c(e)).amax(dim=2) for c in self.convs], dim=1)
        return self.head(h).squeeze(1)

Xall = encode(df.text.tolist())
y = torch.tensor(df.label.values, dtype=torch.float32)
tr_idx, va_idx = np.where(tr)[0], np.where(va)[0]
loader = DataLoader(TensorDataset(Xall[tr_idx], y[tr_idx]), batch_size=256, shuffle=True)
model = CharCNN().to(DEVICE)
opt = torch.optim.AdamW(model.parameters(), lr=2e-3, weight_decay=1e-4)
lossf = nn.BCEWithLogitsLoss()

@torch.no_grad()
def predict(X, bs=1024):
    model.eval()
    out = [torch.sigmoid(model(X[i:i + bs].to(DEVICE))).cpu() for i in range(0, len(X), bs)]
    return torch.cat(out).numpy()

from sklearn.metrics import roc_auc_score
t0 = time.time()
best_auc, best_state = -1, None
for ep in range(1, EPOCHS + 1):
    model.train()
    total = 0.0
    for xb, yb in loader:
        xb, yb = xb.to(DEVICE), yb.to(DEVICE)
        opt.zero_grad()
        loss = lossf(model(xb), yb)
        loss.backward()
        opt.step()
        total += loss.item() * len(xb)
    auc = roc_auc_score(df.label.values[va_idx], predict(Xall[va_idx]))
    print(f'epoch {ep}/{EPOCHS}  loss {total / len(tr_idx):.4f}  val AUC {auc:.4f}')
    if auc > best_auc:
        best_auc, best_state = auc, {k: v.detach().clone() for k, v in model.state_dict().items()}
model.load_state_dict(best_state)
TRAIN_TIME['B_char_cnn'] = round(time.time() - t0, 1)
t0 = time.time()
df['b_score'] = predict(Xall)
B_SPEED = round(len(df) / (time.time() - t0), 1)
print(f'학습 {TRAIN_TIME["B_char_cnn"]}초 ({DEVICE}), 판정 속도 {B_SPEED:,} 요청/초')

## 9. 비교 — 같은 오탐률 기준에서 탐지율 비교
AI 모델은 판정 기준값에 따라 탐지율과 오탐률이 바뀝니다. 공정하게 비교하려고
**검증 집합에서 오탐률(FPR)이 정해진 값이 되는 기준값**을 고른 뒤, 그 기준값으로 시험 집합을 평가합니다.
- FPR 1%, 0.1% : 운영 기준
- "CRS PL1과 같은 FPR" : 룰과 같은 오탐 수준에서 누가 더 잡는가

In [ ]:
from sklearn.metrics import roc_curve

def thr_at_fpr(scores_neg, target):
    """음성(정상) 점수에서 오탐률이 target 이하가 되는 가장 낮은 기준값"""
    s = np.sort(scores_neg)[::-1]
    k = int(np.floor(target * len(s)))
    return s[k] + 1e-9 if k < len(s) else s[-1]

def metrics(name, yt, pred, extra=None):
    yt, pred = np.asarray(yt).astype(int), np.asarray(pred).astype(bool)
    tp = int((pred & (yt == 1)).sum()); fn = int((~pred & (yt == 1)).sum())
    fp = int((pred & (yt == 0)).sum()); tn = int((~pred & (yt == 0)).sum())
    tpr = tp / max(tp + fn, 1); fpr = fp / max(fp + tn, 1); prec = tp / max(tp + fp, 1)
    f1 = 2 * prec * tpr / max(prec + tpr, 1e-9)
    d = dict(model=name, detection_rate=round(tpr, 4), false_positive_rate=round(fpr, 4),
             precision=round(prec, 4), f1=round(f1, 4), TP=tp, FN=fn, FP=fp, TN=tn)
    if extra:
        d.update(extra)
    return d

yv, yt = df.label[va].values, df.label[te].values
rows = []
for pl in (1, 2, 3, 4):
    rows.append(metrics(f'CRS PL{pl}', yt, df[f'crs{pl}_block'][te], {'kind': 'rule', 'speed_rps': CRS_SPEED[f'PL{pl}']}))
crs1_val_fpr = df[f'crs1_block'][va & (df.label == 0)].mean()
targets = {'FPR 1%': 0.01, 'FPR 0.1%': 0.001, f'CRS PL1 FPR ({crs1_val_fpr:.2%})': crs1_val_fpr}
THRESH = {}
for col, nm, spd in (('a_score', 'A TF-IDF+LR', A_SPEED), ('b_score', 'B Char-CNN', B_SPEED)):
    for tname, target in targets.items():
        th = thr_at_fpr(df[col][va & (df.label == 0)].values, target)
        THRESH[f'{nm} @ {tname}'] = float(th)
        rows.append(metrics(f'{nm} @ {tname}', yt, df[col][te] >= th, {'kind': 'ai', 'speed_rps': spd, 'threshold': round(float(th), 6)}))
# 하이브리드: CRS PL1 차단 OR 모델 B(오탐률 0.1% 기준)
thB = THRESH['B Char-CNN @ FPR 0.1%']
df['hybrid'] = df['crs1_block'].eq(True) | (df.b_score >= thB)
rows.append(metrics('Hybrid: CRS PL1 OR B@FPR0.1%', yt, df.hybrid[te], {'kind': 'hybrid'}))
cmp = pd.DataFrame(rows)
pd.set_option('display.width', 200)
print(cmp[['model', 'detection_rate', 'false_positive_rate', 'precision', 'f1', 'TP', 'FN', 'FP', 'TN', 'speed_rps']].to_string(index=False))
cmp.to_csv(os.path.join(RESULTS, 'comparison.csv'), index=False)

## 10. ROC 곡선 — 룰(점)과 AI(곡선)

In [ ]:
import matplotlib
matplotlib.use('Agg') if not IN_COLAB else None
import matplotlib.pyplot as plt
fig, ax = plt.subplots(figsize=(7, 5.5))
for col, nm in (('a_score', 'A TF-IDF+LR'), ('b_score', 'B Char-CNN')):
    f, t, _ = roc_curve(yt, df[col][te])
    ax.plot(f, t, label=f'{nm} (AUC {roc_auc_score(yt, df[col][te]):.3f})')
f, t, _ = roc_curve(yt, df['crs2_score'][te])
ax.plot(f, t, '--', color='gray', label='CRS anomaly score (PL2, varying threshold)')
for pl, mk in zip((1, 2, 3, 4), 'osD^'):
    r = cmp[cmp.model == f'CRS PL{pl}'].iloc[0]
    ax.plot(r.false_positive_rate, r.detection_rate, mk, ms=9, label=f'CRS PL{pl} (block)')
r = cmp[cmp.model.str.startswith('Hybrid')].iloc[0]
ax.plot(r.false_positive_rate, r.detection_rate, '*', ms=14, color='black', label='Hybrid (CRS PL1 OR B)')
ax.set_xscale('symlog', linthresh=0.01); ax.set_xlim(0, 1); ax.set_ylim(0, 1.01)
ax.set_xlabel('False positive rate (log scale above 1%)'); ax.set_ylabel('Detection rate')
ax.set_title('CSIC 2010 test set: rule-based WAF vs AI'); ax.grid(alpha=.3); ax.legend(fontsize=8, loc='lower right')
plt.tight_layout(); plt.savefig(os.path.join(RESULTS, 'roc.png'), dpi=150)
plt.show() if IN_COLAB else None

## 11. 무엇을 놓치나 — CRS 규칙 계열별 분석
시험 집합의 비정상 요청을 **CRS PL1이 반응한 규칙 계열**로 나누고, 계열마다 AI가 얼마나 잡는지 봅니다.
CSIC의 "비정상"에는 공격뿐 아니라 **허용되지 않은 값(파라미터 변조, 이상한 문자)**도 섞여 있어
룰이 전혀 반응하지 않는 요청이 많습니다. 이것이 룰과 AI의 성격 차이를 가장 잘 보여줍니다.

In [ ]:
FAMILY = {'911': 'method', '913': 'scanner', '920': 'protocol', '921': 'protocol-attack', '922': 'multipart',
          '930': 'LFI/path-traversal', '931': 'RFI', '932': 'RCE', '933': 'PHP', '934': 'generic',
          '941': 'XSS', '942': 'SQLi', '943': 'session-fixation', '944': 'Java'}
def family(rules):
    if not isinstance(rules, str) or not rules.strip():
        return '(no CRS match)'
    fams = sorted({FAMILY.get(r[:3], r[:3]) for r in rules.split()})
    return '+'.join(fams)
an = df[te & (df.label == 1)].copy()
an['family'] = an['crs1_rules'].apply(family)
thA1, thB1 = THRESH['A TF-IDF+LR @ FPR 1%'], THRESH['B Char-CNN @ FPR 1%']
fam = an.groupby('family').agg(count=('id', 'size'),
                               crs_pl1_detect=('crs1_block', 'mean'),
                               ai_A_detect=('a_score', lambda s: (s >= thA1).mean()),
                               ai_B_detect=('b_score', lambda s: (s >= thB1).mean())).sort_values('count', ascending=False)
print(fam.round(3).to_string())
fam.to_csv(os.path.join(RESULTS, 'by_crs_family.csv'))

## 12. 사례 — 한쪽만 잡은 요청
운영자 입장에서 가장 중요한 부분입니다. 실제 요청을 보고 왜 갈렸는지 판단합니다.

In [ ]:
pd.set_option('display.max_colwidth', 160)
t = df[te].copy()
t['crs'] = t['crs1_block'].eq(True)
t['ai'] = t.b_score >= thB1
t['request'] = (t.method + ' ' + t.url + ' ' + t.body).str.slice(0, 220)
cases = {
    'ai_only_detected': t[(t.label == 1) & t.ai & ~t.crs],
    'crs_only_detected': t[(t.label == 1) & t.crs & ~t.ai],
    'both_missed': t[(t.label == 1) & ~t.crs & ~t.ai],
    'crs_false_positive': t[(t.label == 0) & t.crs],
    'ai_false_positive': t[(t.label == 0) & t.ai],
}
CASE_COUNTS = {k: int(len(v)) for k, v in cases.items()}
for k, v in cases.items():
    print(f'\n## {k}: {len(v):,}건')
    print(v[['request', 'crs1_rules', 'b_score']].head(6).to_string(index=False))
    v[['id', 'label', 'method', 'url', 'body', 'crs1_rules', 'crs1_score', 'a_score', 'b_score']].head(300).to_csv(
        os.path.join(RESULTS, f'cases_{k}.csv'), index=False)

## 13. 해석할 때 주의할 점
- **CSIC의 "비정상"은 공격만이 아닙니다.** 상당수가 `B1=` 대신 `B1A=`, `id=2%2F`처럼 이 쇼핑몰에서 허용되지 않는 값입니다.
  CRS는 어느 사이트에나 쓰는 **범용 공격 패턴**만 보므로 이런 요청에 반응하지 않는 것이 정상 동작입니다.
- **AI는 "이 사이트의 정상 요청 모양"을 배웠습니다.** 모델 A가 중요하게 본 조각(예: `b1a=`)이 그 증거입니다.
  즉 여기서의 높은 점수는 "이 사이트 전용" 성능이며, **다른 사이트에서도 통한다는 뜻이 아닙니다.**
- 실제 공격 계열(SQLi, XSS, 경로 탐색, 명령 실행)에서는 룰과 AI 모두 거의 다 잡습니다 (11번 표).
- CRS PL3·PL4의 높은 오탐률은 **튜닝(예외 규칙) 없이** 켰을 때의 값입니다. 실제 운영에서는 오탐 규칙을 예외 처리해 낮춥니다.
- 결론의 방향: **룰 = 범용 공격 차단(설명 가능, 학습 불필요)**, **AI = 사이트 전용 이상 탐지(정상 데이터 필요)** — 경쟁이 아니라 보완 관계.

In [ ]:
no_match = float((an.family == '(no CRS match)').mean())
print(f'시험 집합 비정상 요청 중 CRS PL1이 전혀 반응하지 않은 비율: {no_match:.1%}')

## 14. 결과 저장
`results/` 폴더를 압축해 내려받습니다. 포트폴리오 페이지에 쓸 표·그래프가 들어 있습니다.

In [ ]:
summary = {
    'dataset': 'HTTP DATASET CSIC 2010', 'data_stats': DATA_STATS,
    'split': df.groupby(['split', 'label']).size().unstack().to_dict(),
    'versions': {**VERSIONS, 'go': GO_VERSION, 'coraza': CORAZA_VERSION, 'coraza-coreruleset (CRS)': CRS_MODULE_VERSION},
    'train_time_sec': TRAIN_TIME, 'speed_rps': {'crs': CRS_SPEED, 'A': A_SPEED, 'B': B_SPEED},
    'thresholds': THRESH, 'case_counts': CASE_COUNTS, 'anomalous_no_crs_match_share': round(no_match, 4), 'char_cnn_best_val_auc': round(float(best_auc), 4),
    'comparison': cmp.to_dict(orient='records'),
}
json.dump(summary, open(os.path.join(RESULTS, 'summary.json'), 'w'), ensure_ascii=False, indent=2)
torch.save(model.state_dict(), os.path.join(RESULTS, 'char_cnn.pt'))
zip_path = shutil.make_archive(os.path.join(WORK, 'waf_lab_results'), 'zip', RESULTS)
print('저장:', zip_path, f'{os.path.getsize(zip_path)/1e6:.1f} MB')
print(sorted(os.listdir(RESULTS)))
if IN_COLAB:
    from google.colab import files
    files.download(zip_path)